# Notebook 04: Triển Khai Thử Nghiệm Real-time Endpoint & Thu Dọn (Tuần 3)
**Môn học:** Cloud - Đợt 1 - 2026-2027  
**Đề tài:** Phân loại bệnh lá cây bằng Transfer Learning trên SageMaker  
**Mục đích:** 
1. Dựng thử nghiệm một Real-time Endpoint với máy ảo CPU `ml.m5.large` để kiểm tra độ trễ suy luận thời gian thực (Latency ms/ảnh).
2. Gửi ảnh lá cây bệnh thực tế để kiểm chứng phản hồi JSON.
3. **AN TOÀN CHI PHÍ:** Chạy lệnh xóa ngay Endpoint sau 5-10 phút test để tránh phát sinh chi phí duy trì ngoài ý muốn.

In [ ]:
import os
import time
import json
import boto3
import sagemaker
from sagemaker.pytorch.model import PyTorchModel

# 1. Khởi tạo cấu hình môi trường
session = sagemaker.Session()
account_id = boto3.client('sts').get_caller_identity()['Account']
role = f"arn:aws:iam::{account_id}:role/LabRole"
bucket_name = f"sagemaker-plant-leaf-{account_id}"

# Model Artifact MobileNetV2
model_data = f"s3://{bucket_name}/output/pytorch-training-2026-10-18-16-35-53-425/output/model.tar.gz"
endpoint_name = f"plant-leaf-endpoint-{int(time.time())}"

print(f"Account ID     : {account_id}")
print(f"Endpoint Name  : {endpoint_name}")
print(f"Model Artifact : {model_data}")

In [ ]:
# 2. Cấu hình PyTorchModel và Deploy Real-time Endpoint
source_candidates = ['Cloud/src', '../src', 'src']
source_dir = next((d for d in source_candidates if os.path.exists(d)), '../src')

pytorch_model = PyTorchModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    source_dir=source_dir,
    framework_version='2.1.0',
    py_version='py310',
    sagemaker_session=session,
    tags=[
        {'Key': 'Project', 'Value': 'PlantLeafClassification'},
        {'Key': 'Owner', 'Value': 'LeBaTam'}
    ]
)

print(f"Đang triển khai Real-time Endpoint: {endpoint_name} trên instance ml.m5.large...")
print("Quá trình này mất khoảng 3 - 5 phút để AWS nạp container...")

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',       # CPU instance tối ưu chi phí
    endpoint_name=endpoint_name
)

print("=" * 60)
print(f"[THÀNH CÔNG] Endpoint đã sẵn sàng phục vụ: {predictor.endpoint_name}")
print("=" * 60)

In [ ]:
# 3. Gửi ảnh lá cây thực tế để kiểm tra suy luận và đo độ trễ (Latency)
import glob

# Tìm 1 ảnh lá cây mẫu từ thư mục data
sample_images = glob.glob('Cloud/data/*/*.JPG') or glob.glob('../data/*/*.JPG') or glob.glob('data/*/*.JPG')
test_image_path = sample_images[0] if sample_images else None

if test_image_path and os.path.exists(test_image_path):
    print(f"Đang gửi ảnh kiểm thử: {test_image_path}")
    with open(test_image_path, 'rb') as f:
        payload = f.read()

    start_time = time.perf_counter()
    result = predictor.predict(payload, initial_args={'ContentType': 'image/jpeg'})
    latency_ms = (time.perf_counter() - start_time) * 1000

    print("=" * 60)
    print(f"ĐỘ TRỄ SUY LUẬN (LATENCY): {latency_ms:.2f} ms")
    print(f"KẾT QUẢ DỰ ĐOÁN JSON:\n{result}")
    print("=" * 60)
else:
    print("Chưa tìm thấy ảnh mẫu. Vui lòng kiểm tra lại đường dẫn ảnh.")

In [ ]:
# 4. THAO TÁC BẮT BUỘC: XÓA NGAY ENDPOINT ĐỂ TRÁNH PHÁT SINH CHI PHÍ
print("=" * 60)
print(f"Đang thực hiện xóa Endpoint: {predictor.endpoint_name}...")
predictor.delete_endpoint(delete_endpoint_config=True)
predictor.delete_model()
print("[HOÀN TẤT] Endpoint và Endpoint Config đã được xóa sạch sẽ!")
print("Tài khoản an toàn 100%, không bị trừ tiền giờ máy ảo duy trì.")
print("=" * 60)